### Import libraries

In [74]:
import pandas as pd 
import numpy as np 
import os

### Load datasets for inspection and cleaning

#### 1.) Online shop data

In [75]:
shop_data = pd.read_csv(r"C:\Users\Glemba\Desktop\Kenyan retail analysis\00_raw_data\kenyan_online_shop_data.csv")
shop_data.head()

,InvoiceNo,CustomerID,Product,Quantity,UnitPrice,TotalPrice,InvoiceDate,Location,CustomerType
0,ORD10000,CUST1076,Sanitary Towels (Pack),1,153.60,153.60,2/17/2023,Umoja,Regular
1,ORD10001,CUST1075,Bread (400g),1,52.55,52.55,02/09/2023,Nairobi,Regular
2,ORD10002,CUST1020,Milk (500ml),1,44.20,44.20,02/01/2023,Roysambu,Regular
3,ORD10003,CUST1181,Sanitary Towels (Pack),1,250.33,250.33,2/22/2023,Ruiru,Budget
4,ORD10004,CUST1079,Sugar (1kg),1,132.24,132.24,4/13/2023,Nairobi,Budget


In [76]:
def basic_checks(shop_data):
    print('-' * 40, 'Shape', '-' * 40)
    print(shop_data.shape)
    print('-' * 40, 'Columns', '-' * 40)
    print(shop_data.columns)
    print('-' * 40, 'Info', '-' * 40)
    print(shop_data.info())
    print('-' * 40, 'Summary statistics(categorical)', '-' * 40)
    print(display(shop_data.describe(include=['str', 'object']).T))
    print('-' * 40, 'Summary statistics(numerical)', '-' * 40)
    print(display(shop_data.describe(include=['int', 'float']).T))
    print('-' * 40, 'Duplicates', '-' * 40)
    print(shop_data.duplicated().sum())
    print('-' * 40, 'Null Values', '-' * 40)
    print(shop_data.isna().sum())
    print('-' * 40, 'Unique Values', '-' * 40)
    print(shop_data.nunique())
    print('-' * 40, 'Data Types', '-' * 40)
    print(shop_data.dtypes)


In [77]:
basic_checks(shop_data)

---------------------------------------- Shape ----------------------------------------
(1500, 9)
---------------------------------------- Columns ----------------------------------------
Index(['InvoiceNo', 'CustomerID', 'Product', 'Quantity', 'UnitPrice',
       'TotalPrice', 'InvoiceDate', 'Location', 'CustomerType'],
      dtype='str')
---------------------------------------- Info ----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   InvoiceNo     1500 non-null   str    
 1   CustomerID    1500 non-null   str    
 2   Product       1500 non-null   str    
 3   Quantity      1500 non-null   int64  
 4   UnitPrice     1500 non-null   float64
 5   TotalPrice    1500 non-null   float64
 6   InvoiceDate   1500 non-null   str    
 7   Location      1500 non-null   str    
 8   CustomerType  1500 non-null   str    
dty

,count,unique,top,freq
InvoiceNo,1500,1500,ORD10000,1
CustomerID,1500,299,CUST1020,11
Product,1500,10,Eggs (Tray),182
InvoiceDate,1500,271,1/31/2023,24
Location,1500,8,Nairobi,485
CustomerType,1500,3,Budget,921


None
---------------------------------------- Summary statistics(numerical) ----------------------------------------


,count,mean,std,min,25%,50%,75%,max
Quantity,1500.0,1.004667,0.492077,0.00,1.0000,1.00,1.0000,5.00
UnitPrice,1500.0,147.602073,97.613677,29.01,59.5950,124.37,215.2750,366.89
TotalPrice,1500.0,147.188993,116.557819,0.00,57.0925,121.44,217.3975,1152.40


None
---------------------------------------- Duplicates ----------------------------------------
0
---------------------------------------- Null Values ----------------------------------------
InvoiceNo       0
CustomerID      0
Product         0
Quantity        0
UnitPrice       0
TotalPrice      0
InvoiceDate     0
Location        0
CustomerType    0
dtype: int64
---------------------------------------- Unique Values ----------------------------------------
InvoiceNo       1500
CustomerID       299
Product           10
Quantity           6
UnitPrice       1458
TotalPrice      1389
InvoiceDate      271
Location           8
CustomerType       3
dtype: int64
---------------------------------------- Data Types ----------------------------------------
InvoiceNo           str
CustomerID          str
Product             str
Quantity          int64
UnitPrice       float64
TotalPrice      float64
InvoiceDate         str
Location            str
CustomerType        str
dtype: object


In [78]:
### 1) Parse dates and handle missing values
# Convert fields to usable types
shop_data["InvoiceDate"] = pd.to_datetime(
    shop_data["InvoiceDate"],
    format="%m/%d/%Y",
    errors="coerce"
)
shop_data["Quantity"] = pd.to_numeric(shop_data["Quantity"], errors="coerce")
shop_data["UnitPrice"] = pd.to_numeric(shop_data["UnitPrice"], errors="coerce")

# Report and remove rows with invalid dates or numeric values
invalid_rows = shop_data[
    shop_data["InvoiceDate"].isna()
    | shop_data["Quantity"].isna()
    | shop_data["UnitPrice"].isna()
].copy()

print("Rows with invalid dates or numeric values:", len(invalid_rows))

shop_data = shop_data.dropna(
    subset=["InvoiceDate", "Quantity", "UnitPrice"]
).copy()

# Keep zero-quantity rows for review, then exclude them from sales analysis
zero_quantity_rows = shop_data.loc[shop_data["Quantity"].eq(0)].copy()
print("Zero-quantity rows:", len(zero_quantity_rows))

shop_data = shop_data.loc[shop_data["Quantity"].gt(0)].copy()
print("Rows remaining:", len(shop_data))

Rows with invalid dates or numeric values: 0
Zero-quantity rows: 78
Rows remaining: 1422


In [79]:
### 2) Check Quantity and UnitPrice for outliers
def iqr_candidates(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    mask = data[column].lt(lower_fence) | data[column].gt(upper_fence)

    print(
        f"{column}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, "
        f"fences=({lower_fence:.2f}, {upper_fence:.2f}), "
        f"candidates={mask.sum()}"
    )

    return data.loc[
        mask, ["InvoiceNo", "Product", "Location", column]
    ].sort_values(column)


quantity_iqr_candidates = iqr_candidates(shop_data, "Quantity")
unitprice_iqr_candidates = iqr_candidates(shop_data, "UnitPrice")

display(quantity_iqr_candidates)
display(unitprice_iqr_candidates)

Quantity: Q1=1.00, Q3=1.00, IQR=0.00, fences=(1.00, 1.00), candidates=29
UnitPrice: Q1=60.85, Q3=217.50, IQR=156.65, fences=(-174.12, 452.46), candidates=0


,InvoiceNo,Product,Location,Quantity
534,ORD10534,Tea Leaves (250g),Ruaka,2
560,ORD10560,Soap (Bar),Nairobi,2
693,ORD10693,Soap (Bar),Rongai,2
586,ORD10586,Sanitary Towels (Pack),Kitengela,2
1082,ORD11082,Cooking Oil (1L),Thika,2
790,ORD10790,Tea Leaves (250g),Nairobi,3
995,ORD10995,Soap (Bar),Rongai,3
1267,ORD11267,Milk (500ml),Rongai,3
1293,ORD11293,Maize Flour (2kg),Ruaka,3
1050,ORD11050,Rice (1kg),Rongai,4


,InvoiceNo,Product,Location,UnitPrice


#### 2.) Population and housing census

In [80]:
population_data = pd.read_csv(r"C:\Users\Glemba\Desktop\Kenyan retail analysis\00_raw_data\Kenya population and Housing Census Population households density by county (KE1,DF_POP_CENSUS_COU,1.0).csv")
population_data.head()

,Reference Area (KE),Census Indicators (POP_TOT),Sex (M),Employment status (_T),Time period (2019),Observation,Observation Status,Unit of measure,Unit Multipler,Time Period Comment,Time Series Comment,Dataset Comment,Source
0,KENYA,Total Population,Male,_T,2019,23548056,NaN,Number,Units,NaN,NaN,NaN,Kenya National Bureau of Statistics
1,KENYA,Total Population,Female,_T,2019,24014716,NaN,Number,Units,NaN,NaN,NaN,Kenya National Bureau of Statistics
2,KENYA,Total Population,Total,_T,2019,47564296,NaN,Number,Units,NaN,NaN,NaN,Kenya National Bureau of Statistics
3,NAIROBI,Total Population,Male,_T,2019,2192452,NaN,Number,Units,NaN,NaN,NaN,Kenya National Bureau of Statistics
4,NAIROBI,Total Population,Female,_T,2019,2204376,NaN,Number,Units,NaN,NaN,NaN,Kenya National Bureau of Statistics


In [81]:
def basic_checks(population_data):
    print('-' * 40, 'Shape', '-' * 40)
    print(population_data.shape)
    print('-' * 40, 'Columns', '-' * 40)
    print(population_data.columns)
    print('-' * 40, 'Info', '-' * 40)
    print(population_data.info())
    print('-' * 40, 'Summary statistics(categorical)', '-' * 40)
    print(display(population_data.describe(include=['str', 'object']).T))
    print('-' * 40, 'Summary statistics(numerical)', '-' * 40)
    print(display(population_data.describe(include=['int', 'float']).T))
    print('-' * 40, 'Duplicates', '-' * 40)
    print(population_data.duplicated().sum())
    print('-' * 40, 'Null Values', '-' * 40)
    print(population_data.isna().sum())
    print('-' * 40, 'Unique Values', '-' * 40)
    print(population_data.nunique())
    print('-' * 40, 'Data Types', '-' * 40)
    print(population_data.dtypes)

In [82]:
basic_checks(population_data)

---------------------------------------- Shape ----------------------------------------
(42507, 13)
---------------------------------------- Columns ----------------------------------------
Index(['Reference Area (KE)', 'Census Indicators (POP_TOT)', 'Sex (M)',
       'Employment status (_T)', 'Time period (2019)', 'Observation',
       'Observation Status', 'Unit of measure', 'Unit Multipler',
       'Time  Period Comment', 'Time Series Comment', 'Dataset Comment',
       'Source'],
      dtype='str')
---------------------------------------- Info ----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 42507 entries, 0 to 42506
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Reference Area (KE)          42507 non-null  str    
 1   Census Indicators (POP_TOT)  42507 non-null  str    
 2   Sex (M)                      42507 non-null  str    
 3   Employment s

,count,unique,top,freq
Reference Area (KE),42507,9789,TOWNSHIP,108
Census Indicators (POP_TOT),42507,1,Total Population,42507
Sex (M),42507,3,Male,14169
Employment status (_T),42507,1,_T,42507
Unit of measure,42507,1,Number,42507
Unit Multipler,42507,1,Units,42507
Source,42507,1,Kenya National Bureau of Statistics,42507


None
---------------------------------------- Summary statistics(numerical) ----------------------------------------


,count,mean,std,min,25%,50%,75%,max
Time period (2019),42507.0,2019.000000,0.000000,2019.0,2019.0,2019.0,2019.0,2019.0
Observation,42507.0,13427.492131,287576.046054,0.0,1594.0,3138.0,7078.5,47564296.0
Observation Status,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time Period Comment,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time Series Comment,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dataset Comment,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


None
---------------------------------------- Duplicates ----------------------------------------
2016
---------------------------------------- Null Values ----------------------------------------
Reference Area (KE)                0
Census Indicators (POP_TOT)        0
Sex (M)                            0
Employment status (_T)             0
Time period (2019)                 0
Observation                        0
Observation Status             42507
Unit of measure                    0
Unit Multipler                     0
Time  Period Comment           42507
Time Series Comment            42507
Dataset Comment                42507
Source                             0
dtype: int64
---------------------------------------- Unique Values ----------------------------------------
Reference Area (KE)             9789
Census Indicators (POP_TOT)        1
Sex (M)                            3
Employment status (_T)             1
Time period (2019)                 1
Observation                 

#### cleaning

In [83]:
COUNTIES = ["MOMBASA","KWALE","KILIFI","TANA RIVER","LAMU","TAITA/TAVETA","GARISSA","WAJIR","MANDERA","MARSABIT",
 "ISIOLO","MERU","THARAKA NITHI","EMBU","KITUI","MACHAKOS","MAKUENI","NYANDARUA","NYERI","KIRINYAGA","MURANGA","KIAMBU",
 "TURKANA","WEST POKOT","SAMBURU","TRANS NZOIA","UASIN GISHU","ELGEYO/MARAKWET","NANDI","BARINGO","LAIKIPIA","NAKURU",
 "NAROK","KAJIADO","KERICHO","BOMET","KAKAMEGA","VIHIGA","BUNGOMA","BUSIA","SIAYA","KISUMU","HOMA BAY","MIGORI","KISII",
 "NYAMIRA","NAIROBI"]

def clean_population(df):
    n = len(df)
    df = df.drop_duplicates()
    print(f"Removed {n - len(df)} duplicates")
    df = (df.rename(columns={"Reference Area (KE)": "Area", "Sex (M)": "Sex", "Observation": "Population"})
            [["Area", "Sex", "Population"]])
    df["Area"] = df["Area"].str.strip().str.upper()
    return df


population_clean = clean_population(population_data)   # assign it this time

# one row per county (names repeat as wards, so keep the largest = the county itself)
county_population = (
    population_clean[population_clean["Area"].isin(COUNTIES)]
    .pivot_table(index="Area", columns="Sex", values="Population", aggfunc="max")
    .rename(columns={"Male": "MalePopulation", "Female": "FemalePopulation", "Total": "TotalPopulation"})
    .rename_axis("County").reset_index()
)
assert len(county_population) == 47, len(county_population)

county_population.head()


Removed 2016 duplicates


Sex,County,FemalePopulation,MalePopulation,TotalPopulation
0,BARINGO,330428,336322,666763
1,BOMET,441379,434287,875689
2,BUNGOMA,858389,812146,1670570
3,BUSIA,467401,426252,893681
4,ELGEYO/MARAKWET,227151,227317,454480


#### 3.) inflation rates

In [84]:
inflation_data = pd.read_csv(r"C:\Users\Glemba\Desktop\Kenyan retail analysis\00_raw_data\kenya_cpi_inflation_rates_2009-01-01_to_2026-09-01.csv")
inflation_data.head()

,Period,Series,Value,Monthly Change (%),Annual Change (%),Category,Region,Data Type
0,2010-01-01,Overall Inflation (National),7.525587,0.224467,7.525587,Overall Inflation,National,Inflation Rate
1,2010-02-01,Overall Inflation (National),5.168363,0.261292,5.168363,Overall Inflation,National,Inflation Rate
2,2010-03-01,Overall Inflation (National),3.975179,-0.186150,3.975179,Overall Inflation,National,Inflation Rate
3,2010-04-01,Overall Inflation (National),3.652441,0.559493,3.652441,Overall Inflation,National,Inflation Rate
4,2010-05-01,Overall Inflation (National),3.863899,0.204006,3.863899,Overall Inflation,National,Inflation Rate


In [85]:
def basic_checks(inflation_data):
    print('-' * 40, 'Shape', '-' * 40)
    print(inflation_data.shape)
    print('-' * 40, 'Columns', '-' * 40)
    print(inflation_data.columns)
    print('-' * 40, 'Info', '-' * 40)
    print(inflation_data.info())
    print('-' * 40, 'Summary statistics(categorical)', '-' * 40)
    print(display(inflation_data.describe(include=['str', 'object']).T))
    print('-' * 40, 'Summary statistics(numerical)', '-' * 40)
    print(display(inflation_data.describe(include=['int', 'float']).T))
    print('-' * 40, 'Duplicates', '-' * 40)
    print(inflation_data.duplicated().sum())
    print('-' * 40, 'Null Values', '-' * 40)
    print(inflation_data.isna().sum())
    print('-' * 40, 'Unique Values', '-' * 40)
    print(inflation_data.nunique())
    print('-' * 40, 'Data Types', '-' * 40)
    print(inflation_data.dtypes)

In [86]:
basic_checks(inflation_data)

---------------------------------------- Shape ----------------------------------------
(201, 8)
---------------------------------------- Columns ----------------------------------------
Index(['Period', 'Series', 'Value', 'Monthly Change (%)', 'Annual Change (%)',
       'Category', 'Region', 'Data Type'],
      dtype='str')
---------------------------------------- Info ----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Period              201 non-null    str    
 1   Series              201 non-null    str    
 2   Value               201 non-null    float64
 3   Monthly Change (%)  201 non-null    float64
 4   Annual Change (%)   201 non-null    float64
 5   Category            201 non-null    str    
 6   Region              201 non-null    str    
 7   Data Type           201 non-null    str    
dt

,count,unique,top,freq
Period,201,201,2010-01-01,1
Series,201,1,Overall Inflation (National),201
Category,201,1,Overall Inflation,201
Region,201,1,National,201
Data Type,201,1,Inflation Rate,201


None
---------------------------------------- Summary statistics(numerical) ----------------------------------------


,count,mean,std,min,25%,50%,75%,max
Value,201.0,6.621342,3.121779,2.728403,4.508043,6.086719,7.218132,19.709619
Monthly Change (%),201.0,0.537419,0.585660,-1.199917,0.221223,0.483158,0.893437,3.193851
Annual Change (%),201.0,6.621342,3.121779,2.728403,4.508043,6.086719,7.218132,19.709619


None
---------------------------------------- Duplicates ----------------------------------------
0
---------------------------------------- Null Values ----------------------------------------
Period                0
Series                0
Value                 0
Monthly Change (%)    0
Annual Change (%)     0
Category              0
Region                0
Data Type             0
dtype: int64
---------------------------------------- Unique Values ----------------------------------------
Period                201
Series                  1
Value                 201
Monthly Change (%)    201
Annual Change (%)     201
Category                1
Region                  1
Data Type               1
dtype: int64
---------------------------------------- Data Types ----------------------------------------
Period                    str
Series                    str
Value                 float64
Monthly Change (%)    float64
Annual Change (%)     float64
Category                  str
Region     

#### 4.) CPI values

In [87]:
cpi = pd.read_csv(r"C:\Users\Glemba\Desktop\Kenyan retail analysis\00_raw_data\kenya_cpi_values_2009-01-01_to_2026-09-01.csv")
cpi.head()

,Period,Series,Value,Monthly Change (%),Annual Change (%),Category,Region,Data Type
0,2009-01-01,Overall CPI (National),49.83,NaN,NaN,Overall CPI,National,Consumer Price Index
1,2009-02-01,Overall CPI (National),51.08,2.508529,NaN,Overall CPI,National,Consumer Price Index
2,2009-03-01,Overall CPI (National),51.57,0.959280,NaN,Overall CPI,National,Consumer Price Index
3,2009-04-01,Overall CPI (National),52.02,0.872600,NaN,Overall CPI,National,Consumer Price Index
4,2009-05-01,Overall CPI (National),52.02,0.000000,NaN,Overall CPI,National,Consumer Price Index


In [88]:
def basic_checks(cpi):
    print('-' * 40, 'Shape', '-' * 40)
    print(cpi.shape)
    print('-' * 40, 'Columns', '-' * 40)
    print(cpi.columns)
    print('-' * 40, 'Info', '-' * 40)
    print(cpi.info())
    print('-' * 40, 'Summary statistics(categorical)', '-' * 40)
    print(display(cpi.describe(include=['str', 'object']).T))
    print('-' * 40, 'Summary statistics(numerical)', '-' * 40)
    print(display(cpi.describe(include=['int', 'float']).T))
    print('-' * 40, 'Duplicates', '-' * 40)
    print(cpi.duplicated().sum())
    print('-' * 40, 'Null Values', '-' * 40)
    print(cpi.isna().sum())
    print('-' * 40, 'Unique Values', '-' * 40)
    print(cpi.nunique())
    print('-' * 40, 'Data Types', '-' * 40)
    print(cpi.dtypes)

In [89]:
basic_checks(cpi)

---------------------------------------- Shape ----------------------------------------
(213, 8)
---------------------------------------- Columns ----------------------------------------
Index(['Period', 'Series', 'Value', 'Monthly Change (%)', 'Annual Change (%)',
       'Category', 'Region', 'Data Type'],
      dtype='str')
---------------------------------------- Info ----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 213 entries, 0 to 212
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Period              213 non-null    str    
 1   Series              213 non-null    str    
 2   Value               213 non-null    float64
 3   Monthly Change (%)  212 non-null    float64
 4   Annual Change (%)   201 non-null    float64
 5   Category            213 non-null    str    
 6   Region              213 non-null    str    
 7   Data Type           213 non-null    str    
dt

,count,unique,top,freq
Period,213,213,2009-01-01,1
Series,213,1,Overall CPI (National),213
Category,213,1,Overall CPI,213
Region,213,1,National,213
Data Type,213,1,Consumer Price Index,213


None
---------------------------------------- Summary statistics(numerical) ----------------------------------------


,count,mean,std,min,25%,50%,75%,max
Value,213.0,97.388779,30.582359,49.830000,71.300000,94.740000,122.170000,156.470000
Monthly Change (%),212.0,0.542919,0.590189,-1.199917,0.220943,0.479312,0.892104,3.193851
Annual Change (%),201.0,6.621342,3.121779,2.728403,4.508043,6.086719,7.218132,19.709619


None
---------------------------------------- Duplicates ----------------------------------------
0
---------------------------------------- Null Values ----------------------------------------
Period                 0
Series                 0
Value                  0
Monthly Change (%)     1
Annual Change (%)     12
Category               0
Region                 0
Data Type              0
dtype: int64
---------------------------------------- Unique Values ----------------------------------------
Period                213
Series                  1
Value                 211
Monthly Change (%)    211
Annual Change (%)     201
Category                1
Region                  1
Data Type               1
dtype: int64
---------------------------------------- Data Types ----------------------------------------
Period                    str
Series                    str
Value                 float64
Monthly Change (%)    float64
Annual Change (%)     float64
Category                  str
Reg

#### 5.) Public holidays

In [90]:
holidays = pd.read_excel(r"C:\Users\Glemba\Desktop\Kenyan retail analysis\00_raw_data\kenya_public_holidays_2020_2026.xlsx")
holidays.head()

,Year,Holiday Name,Date,Day of Week,Type
0,2020,New Year's Day,2020-01-01,Wednesday,National
1,2020,Good Friday,2020-04-10,Friday,Religious
2,2020,Easter Monday,2020-04-13,Monday,Religious
3,2020,Labour Day,2020-05-01,Friday,National
4,2020,Idd-ul-Fitr,2020-05-25,Monday,Religious


In [91]:
def basic_checks(holidays):
    print('-' * 40, 'Shape', '-' * 40)
    print(holidays.shape)
    print('-' * 40, 'Columns', '-' * 40)
    print(holidays.columns)
    print('-' * 40, 'Info', '-' * 40)
    print(holidays.info())
    print('-' * 40, 'Summary statistics(categorical)', '-' * 40)
    print(display(holidays.describe(include=['str', 'object']).T))
    print('-' * 40, 'Summary statistics(numerical)', '-' * 40)
    print(display(holidays.describe(include=['int', 'float']).T))
    print('-' * 40, 'Duplicates', '-' * 40)
    print(holidays.duplicated().sum())
    print('-' * 40, 'Null Values', '-' * 40)
    print(holidays.isna().sum())
    print('-' * 40, 'Unique Values', '-' * 40)
    print(holidays.nunique())
    print('-' * 40, 'Data Types', '-' * 40)
    print(holidays.dtypes)

In [92]:
basic_checks(holidays)

---------------------------------------- Shape ----------------------------------------
(84, 5)
---------------------------------------- Columns ----------------------------------------
Index(['Year', 'Holiday Name', 'Date', 'Day of Week', 'Type'], dtype='str')
---------------------------------------- Info ----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Year          84 non-null     int64
 1   Holiday Name  84 non-null     str  
 2   Date          84 non-null     str  
 3   Day of Week   84 non-null     str  
 4   Type          84 non-null     str  
dtypes: int64(1), str(4)
memory usage: 3.4 KB
None
---------------------------------------- Summary statistics(categorical) ----------------------------------------


,count,unique,top,freq
Holiday Name,84,14,New Year's Day,7
Date,84,84,2020-01-01,1
Day of Week,84,7,Friday,20
Type,84,2,National,56


None
---------------------------------------- Summary statistics(numerical) ----------------------------------------


,count,mean,std,min,25%,50%,75%,max
Year,84.0,2023.0,2.012012,2020.0,2021.0,2023.0,2025.0,2026.0


None
---------------------------------------- Duplicates ----------------------------------------
0
---------------------------------------- Null Values ----------------------------------------
Year            0
Holiday Name    0
Date            0
Day of Week     0
Type            0
dtype: int64
---------------------------------------- Unique Values ----------------------------------------
Year             7
Holiday Name    14
Date            84
Day of Week      7
Type             2
dtype: int64
---------------------------------------- Data Types ----------------------------------------
Year            int64
Holiday Name      str
Date              str
Day of Week       str
Type              str
dtype: object


In [93]:
### 3) Build the features to use later for EDA and more 
# Calendar features
shop_data["InvoiceMonth"] = shop_data["InvoiceDate"].dt.month
shop_data["InvoiceWeekday"] = shop_data["InvoiceDate"].dt.day_name()
shop_data["InvoiceWeekdayNum"] = shop_data["InvoiceDate"].dt.dayofweek

# Public-holiday flag
holiday_dates = pd.to_datetime(
    holidays["Date"], errors="coerce"
).dt.normalize().dropna()

shop_data["IsHoliday"] = (
    shop_data["InvoiceDate"].dt.normalize()
    .isin(holiday_dates)
    .astype("int8")
)

# Revenue for each invoice line
shop_data["LineRevenue"] = (
    shop_data["Quantity"] * shop_data["UnitPrice"]
).round(2)

# Map each town/location to its county
town_to_county = {
    "kitengela": "Kajiado",
    "nairobi": "Nairobi",
    "rongai": "Kajiado",
    "roysambu": "Nairobi",
    "ruaka": "Kiambu",
    "ruiru": "Kiambu",
    "thika": "Kiambu",
    "umoja": "Nairobi",
}

location_key = shop_data["Location"].astype("string").str.strip().str.lower()
shop_data["County"] = location_key.map(town_to_county)

unmapped_locations = sorted(
    shop_data.loc[shop_data["County"].isna(), "Location"]
    .dropna()
    .unique()
    .tolist()
)

if unmapped_locations:
    raise ValueError(f"Add county mappings for: {unmapped_locations}")

shop_data["County"] = shop_data["County"].str.upper()

shop_data.head()

,InvoiceNo,CustomerID,Product,Quantity,UnitPrice,TotalPrice,InvoiceDate,Location,CustomerType,InvoiceMonth,InvoiceWeekday,InvoiceWeekdayNum,IsHoliday,LineRevenue,County
0,ORD10000,CUST1076,Sanitary Towels (Pack),1,153.60,153.60,2023-02-17,Umoja,Regular,2,Friday,4,0,153.60,NAIROBI
1,ORD10001,CUST1075,Bread (400g),1,52.55,52.55,2023-02-09,Nairobi,Regular,2,Thursday,3,0,52.55,NAIROBI
2,ORD10002,CUST1020,Milk (500ml),1,44.20,44.20,2023-02-01,Roysambu,Regular,2,Wednesday,2,0,44.20,NAIROBI
3,ORD10003,CUST1181,Sanitary Towels (Pack),1,250.33,250.33,2023-02-22,Ruiru,Budget,2,Wednesday,2,0,250.33,KIAMBU
4,ORD10004,CUST1079,Sugar (1kg),1,132.24,132.24,2023-04-13,Nairobi,Budget,4,Thursday,3,0,132.24,NAIROBI


### Export cleaned data

In [94]:
# define target folder and file path
output_folder = r"C:\Users\Glemba\Desktop\Kenyan retail analysis\01_cleaned_data"

files = {
    "shop_data.csv": shop_data,
    "zero_quantity_rows.csv": zero_quantity_rows,
    "population_data.csv": population_data,
    "county_population.csv": county_population,
    "inflation_data.csv": inflation_data,
    "cpi_data.csv": cpi,
    "holidays_data.csv": holidays
}

for file_name, df in files.items():
    file_path = os.path.join(output_folder, file_name)
    df.to_csv(file_path, index=False)
    print(f"Saved {file_name} to {file_path}")

Saved shop_data.csv to C:\Users\Glemba\Desktop\Kenyan retail analysis\01_cleaned_data\shop_data.csv
Saved zero_quantity_rows.csv to C:\Users\Glemba\Desktop\Kenyan retail analysis\01_cleaned_data\zero_quantity_rows.csv


Saved population_data.csv to C:\Users\Glemba\Desktop\Kenyan retail analysis\01_cleaned_data\population_data.csv
Saved county_population.csv to C:\Users\Glemba\Desktop\Kenyan retail analysis\01_cleaned_data\county_population.csv
Saved inflation_data.csv to C:\Users\Glemba\Desktop\Kenyan retail analysis\01_cleaned_data\inflation_data.csv
Saved cpi_data.csv to C:\Users\Glemba\Desktop\Kenyan retail analysis\01_cleaned_data\cpi_data.csv
Saved holidays_data.csv to C:\Users\Glemba\Desktop\Kenyan retail analysis\01_cleaned_data\holidays_data.csv
